## silver

In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, functions as f
from functools import reduce

### schema creation

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalogmeteo.silver;

take the tables from the bronze layer

In [0]:
weather_milan_24 = spark.read.table(f"catalogmeteo.bronze.weather_milan_2024")
weather_rome_24 = spark.read.table(f"catalogmeteo.bronze.weather_rome_2024")
weather_stations_24 = spark.read.table(f"catalogmeteo.bronze.weather_stations_metadata")

## fix the schema

### for every table that involves observations such that negative temperatures are read properly

In [0]:
# Define the regex pattern that worked
clean_pattern = r"[^0-9.\-]"
numeric_string_cols = {"temperature_c", "feels_like_c"}

# Iterate through all variables in the current notebook environment
for var_name, var_value in list(globals().items()):
    # Check if the variable is a DataFrame and doesn't contain "stations" in its name
    if isinstance(var_value, DataFrame) and "stations" not in var_name.lower():
        
        # Identify columns that are strings and need numeric cleaning
        target_cols = [c for c, t in var_value.dtypes if t == "string" and c in numeric_string_cols]
        
        for col_name in target_cols:
            var_value = var_value.withColumn(
                col_name,
                f.regexp_replace(f.col(col_name), clean_pattern, "").cast("double")
            )
        
        # Update the variable in the global namespace with the cleaned DataFrame
        globals()[var_name] = var_value

### union of observations and drop irrelevant columns

In [0]:
# 1. Gather all the DataFrames you want to union into a list
# We use the same logic as before to filter out "stations"
dfs_to_union = [
    var_value for var_name, var_value in globals().items() 
    if isinstance(var_value, DataFrame) and "stations" not in var_name.lower()
]

# 2. Apply the union using reduce
if dfs_to_union:
    weather_final_union = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), dfs_to_union)  
    # 3. Clean up the dropped columns in one go
    weather_final_union = weather_final_union.drop("uv_index", "_rescued_data")
else:
    print("No DataFrames found to union.")

In [0]:
weather_stations_24 = weather_stations_24.drop("_rescued_data")

## save the united observations in the silver layer

In [0]:
# unified observations + station info
table_name = "catalogmeteo.silver.weather_observations_24"

if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)
    (
        delta_table.alias("target")
        .merge(
            weather_union_24.alias("source"),
            "target.station_id = source.station_id AND target.date = source.date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    weather_union_24.write.format("delta").saveAsTable(table_name)